### Naïve Baselines for Day-Ahead Prices

We include three seasonal naïve baselines as simple, training-free references (per region):
- Naïve¹: use the previous day's 24 prices.
- Naïve²: average the past 3 days' 24 prices (per hour).
- Naïve³: average the past 7 days' 24 prices (per hour).

For probabilistic outputs, we compute empirical quantiles per delivery hour h ∈ {0,…,23} across all test samples of the baseline predictions:
- Q0.1[h], Q0.5[h], Q0.9[h]

We then evaluate using:
- AQL: average pinball loss across the three quantiles
- RMSE, MAE, R² using the median quantile Q0.5 as the point forecast.

Notes
- No models are trained; this is pure calculation on the historical price series.
- The test split follows `Main.py` (2022‑01‑01 | 2024‑01‑01 | 2024‑07‑01 | 2025‑01‑01).

In [ ]:
# Imports and common setup
import os, pickle, math
import numpy as np
import pandas as pd
from pathlib import Path

# Reuse metrics helpers from the project if available; otherwise define minimal versions
try:
    from PriceFM import pinball_loss
except Exception:
    def pinball_loss(y_true, y_pred, q):
        e = y_true - y_pred
        return np.mean(np.maximum(q*e, (q-1)*e))

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pre_path = os.path.abspath('.') + os.sep
data_csv = os.path.join(pre_path, 'Data', 'EU_Spatiotemporal_Energy_Data.csv')
assert os.path.exists(data_csv), f'Missing dataset: {data_csv}'

# Align with Main.py
QUANTILES = [0.1, 0.5, 0.9]
TRAIN_START, VAL_START, TEST_START, TEST_END = ('2022-01-01','2024-01-01','2024-07-01','2025-01-01')

# Regions (same ordering as Main.py)
ALL_REGIONS = ['AT','BE','BG','CZ','DE_LU','DK_1','DK_2','EE','ES','FI','FR','GR','HR','HU','IT_1','IT_2','IT_3','IT_4','IT_5','IT_6','IT_7','LT','LV','NL','NO_1','NO_2','NO_3','NO_4','NO_5','PL','PT','RO','SE_1','SE_2','SE_3','SE_4','SI','SK']

df = pd.read_csv(data_csv)
df['CET'] = pd.to_datetime(df['CET'], utc=True).dt.tz_convert('CET')
df = df.sort_values('CET').ffill().bfill()
df = df.set_index('CET', drop=False)

# Split
train_df = df[(df['CET']>=TRAIN_START) & (df['CET']<VAL_START)]
val_df   = df[(df['CET']>=VAL_START) & (df['CET']<TEST_START)]
test_df  = df[(df['CET']>=TEST_START) & (df['CET']<TEST_END)]
assert len(test_df)>0, 'Empty test split'

# Build region->price series
def price_col(region):
    # target column contains 'DA_price' and region code
    cols = [c for c in df.columns if ('DA_price' in c and region in c)]
    assert len(cols)==1, f'Ambiguous price column for {region}: {cols}'
    return cols[0]

In [2]:
# Utilities to form daily targets Y (24h) and naïve predictions
def daily_targets(series):
    # Returns array of shape (n_days, 24) for consecutive UTC-normalized CET days in test split
    s = series.copy()
    # ensure hourly frequency
    s = s.asfreq('H')
    # slice test period
    s = s.loc[(s.index>=pd.Timestamp(TEST_START, tz='CET')) & (s.index<pd.Timestamp(TEST_END, tz='CET'))]
    # group by date and pivot to 24 columns (0..23)
    g = s.groupby(s.index.date)
    days = []
    for d, vs in g:
        vs = vs.sort_index()
        # require 24 hours
        if len(vs)>=24:
            # take first 24 hours from 00:00 to 23:00
            first = vs.index[0]
            if first.hour!=0:
                # align to day start if possible
                day_start = pd.Timestamp(year=first.year, month=first.month, day=first.day, tz='CET')
                vs = vs.loc[day_start:day_start+pd.Timedelta(hours=23)]
            else:
                vs = vs.iloc[:24]
            if len(vs)==24:
                days.append(vs.values)
    return np.array(days)

def naive_prev_day(series):
    # For each day D in test, prediction is the price of day D-1 (same hour).
    s = series.copy().asfreq('H')
    test_days = pd.date_range(TEST_START, TEST_END, tz='CET', freq='D')[:-1]
    preds = []
    for day in test_days:
        prev = day - pd.Timedelta(days=1)
        y = s.loc[day:day+pd.Timedelta(hours=23)]
        x = s.loc[prev:prev+pd.Timedelta(hours=23)]
        if len(y)==24 and len(x)==24:
            preds.append(x.values)
    return np.array(preds)

def naive_kday_avg(series, k):
    # For each day D, average previous k days by hour
    s = series.copy().asfreq('H')
    test_days = pd.date_range(TEST_START, TEST_END, tz='CET', freq='D')[:-1]
    preds = []
    for day in test_days:
        ok = True
        stack = []
        for i in range(1, k+1):
            ref = day - pd.Timedelta(days=i)
            x = s.loc[ref:ref+pd.Timedelta(hours=23)]
            if len(x)!=24:
                ok=False; break
            stack.append(x.values)
        if ok:
            preds.append(np.mean(stack, axis=0))
    return np.array(preds)

def empirical_hourly_quantiles(preds, qs=(0.1,0.5,0.9)):
    # preds: (n_days, 24)
    if preds.size==0:
        return {q: np.full(24, np.nan) for q in qs}
    return {q: np.quantile(preds, q, axis=0) for q in qs}

def metrics_from_point(y_true_24, y_pred_24):
    # y_true_24, y_pred_24: (n_days, 24)
    yt = y_true_24.reshape(-1)
    yp = y_pred_24.reshape(-1)
    rmse = math.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    return rmse, mae, r2

def avg_pinball(y_true_24, qpred_24_dict, qs=(0.1,0.5,0.9)):
    # For AQL, use per-hour quantile predictions (constants across days) against all true values
    yt = y_true_24  # (n_days, 24)
    losses = []
    for q in qs:
        qh = qpred_24_dict[q]  # (24,)
        # broadcast to (n_days, 24)
        qp = np.broadcast_to(qh, yt.shape)
        losses.append(pinball_loss(yt, qp, q))
    return float(np.mean(losses))

In [6]:
# Main computation per region
results_rows = []
for region in ALL_REGIONS:
    col = price_col(region)
    series = df.set_index('CET')[col]
    # true daily targets (n_days, 24)
    y_true_24 = daily_targets(series)
    # naive predictions (per-day, 24)
    p1 = naive_prev_day(series)                # Naïve¹
    p2 = naive_kday_avg(series, k=3)           # Naïve²
    p3 = naive_kday_avg(series, k=7)           # Naïve³

    # To compare, align by min length
    n = min(len(y_true_24), len(p1), len(p2), len(p3))
    if n == 0:
        continue
    y_true_24 = y_true_24[:n]
    p1, p2, p3 = p1[:n], p2[:n], p3[:n]

    # empirical hour-wise quantiles for each baseline (probabilistic predictions)
    q1 = empirical_hourly_quantiles(p1, qs=tuple(QUANTILES))
    q2 = empirical_hourly_quantiles(p2, qs=tuple(QUANTILES))
    q3 = empirical_hourly_quantiles(p3, qs=tuple(QUANTILES))

    # Quantile losses (pinball) per quantile using per-hour quantile predictions broadcast across days
    # Naïve¹
    ql1_01 = float(pinball_loss(y_true_24, np.broadcast_to(q1[0.1], y_true_24.shape), 0.1))
    ql1_05 = float(pinball_loss(y_true_24, np.broadcast_to(q1[0.5], y_true_24.shape), 0.5))
    ql1_09 = float(pinball_loss(y_true_24, np.broadcast_to(q1[0.9], y_true_24.shape), 0.9))
    aql1 = float(np.mean([ql1_01, ql1_05, ql1_09]))

    # Naïve²
    ql2_01 = float(pinball_loss(y_true_24, np.broadcast_to(q2[0.1], y_true_24.shape), 0.1))
    ql2_05 = float(pinball_loss(y_true_24, np.broadcast_to(q2[0.5], y_true_24.shape), 0.5))
    ql2_09 = float(pinball_loss(y_true_24, np.broadcast_to(q2[0.9], y_true_24.shape), 0.9))
    aql2 = float(np.mean([ql2_01, ql2_05, ql2_09]))

    # Naïve³
    ql3_01 = float(pinball_loss(y_true_24, np.broadcast_to(q3[0.1], y_true_24.shape), 0.1))
    ql3_05 = float(pinball_loss(y_true_24, np.broadcast_to(q3[0.5], y_true_24.shape), 0.5))
    ql3_09 = float(pinball_loss(y_true_24, np.broadcast_to(q3[0.9], y_true_24.shape), 0.9))
    aql3 = float(np.mean([ql3_01, ql3_05, ql3_09]))

    # Point forecast via median quantile per hour; broadcast over all days
    med1 = np.broadcast_to(q1[0.5], y_true_24.shape)
    med2 = np.broadcast_to(q2[0.5], y_true_24.shape)
    med3 = np.broadcast_to(q3[0.5], y_true_24.shape)

    rmse1, mae1, r2_1 = metrics_from_point(y_true_24, med1)
    rmse2, mae2, r2_2 = metrics_from_point(y_true_24, med2)
    rmse3, mae3, r2_3 = metrics_from_point(y_true_24, med3)

    # Store summaries: Q0.1/Q0.5/Q0.9 now mean quantile losses, and AQL is their average
    results_rows.append({
        'region': region,
        'model': 'Naive^1',
        'Q0.1': ql1_01, 'Q0.5': ql1_05, 'Q0.9': ql1_09,
        'AQL': aql1, 'RMSE': rmse1, 'MAE': mae1, 'R2': r2_1,
    })
    results_rows.append({
        'region': region,
        'model': 'Naive^2',
        'Q0.1': ql2_01, 'Q0.5': ql2_05, 'Q0.9': ql2_09,
        'AQL': aql2, 'RMSE': rmse2, 'MAE': mae2, 'R2': r2_2,
    })
    results_rows.append({
        'region': region,
        'model': 'Naive^3',
        'Q0.1': ql3_01, 'Q0.5': ql3_05, 'Q0.9': ql3_09,
        'AQL': aql3, 'RMSE': rmse3, 'MAE': mae3, 'R2': r2_3,
    })

results_df = pd.DataFrame(results_rows)
results_df.head()

,region,model,Q0.1,Q0.5,Q0.9,AQL,RMSE,MAE,R2
0,AT,Naive^1,7.227525,15.792885,8.904050,10.641486,53.669545,31.585770,0.125125
1,AT,Naive^2,7.428931,16.016721,8.980306,10.808653,54.201580,32.033443,0.107693
2,AT,Naive^3,7.781091,16.257513,9.149782,11.062795,54.813336,32.515026,0.087437
3,BE,Naive^1,7.121766,15.586985,7.169700,9.959484,44.804767,31.173970,0.154026
4,BE,Naive^2,7.409020,15.782560,7.235338,10.142306,45.055232,31.565121,0.144541


In [7]:
# Aggregate view: per-model averages across regions
agg = results_df.groupby('model')[['Q0.1','Q0.5','Q0.9','AQL','RMSE','MAE','R2']].mean().reset_index()
agg

,model,Q0.1,Q0.5,Q0.9,AQL,RMSE,MAE,R2
0,Naive^1,5.679329,14.299345,8.308501,9.429058,46.899504,28.598691,0.158239
1,Naive^2,6.012084,14.420198,8.438281,9.623521,46.728548,28.840397,0.165020
2,Naive^3,6.649652,14.575332,8.713225,9.979403,46.726731,29.150664,0.165180


In [ ]:
# Optional: save results
out_dir = os.path.join(pre_path, 'Result')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f'{TRAIN_START}_{VAL_START}_{TEST_START}_naives.csv')
results_df.to_csv(out_path, index=False)
print('Saved:', out_path)